In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import CIFAR10, CIFAR100, MNIST, STL10
import torchvision.transforms as T
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mhnlib.utils as mhn_utils
from math import log, sqrt
import seaborn as sns
from tqdm.auto import tqdm
from pathlib import Path
import re
from einops import rearrange
import pandas as pd
import networkx as nx
import glasbey
import matplotlib.colors as mcolors
from sklearn.decomposition import PCA
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
#device = torch.device("cpu")

In [ ]:
from numba import njit
@njit(nogil=True)
def distance_tree(data_list, threshold : float):
    edges_from = []
    edges_to = []
    num_levels = len(data_list)
    for level in range(0, num_levels-1):
        num_elements_level = len(data_list[level])
        num_elements_next_level = len(data_list[level+1])
        
        for idx in range(0, num_elements_level):
            sq_norm_i = np.sum(data_list[level][idx]**2)
            num_children_found = 0
            for jdx in range(0, num_elements_next_level):
                sq_norm_j = np.sum(data_list[level+1][jdx]**2)
                dist_ij = sq_norm_i + sq_norm_j - 2*np.sum(data_list[level][idx]*data_list[level+1][jdx])
                if dist_ij <= threshold:
                    edges_from.append((level, idx))
                    edges_to.append((level+1, jdx))
                    num_children_found += 1
    return np.array(edges_from), np.array(edges_to)
def disambiguate_fixed_points(x_fp_list, weights, threshold, non_decreasing):
    pbar = tqdm(range(len(x_fp_list)))
    num_fps = [ ]
    x_fps = []
    w_fps = []
    for idx in pbar:
        x_unique, x_unique_counts, labels = mhn_utils.group_by_distance(x_fp_list[idx], threshold)
        w_unique = torch.zeros(x_unique.shape[0], weights.shape[-1])
        w_unique.index_add_(0, labels, weights[idx])
        w_unique /= x_unique_counts[:, None]
        num_fps.append(x_unique.shape[0])
        x_fps.append(torch.as_tensor(x_unique))
        w_fps.append(w_unique)
        if len(num_fps) > 1 and non_decreasing:
            if num_fps[-1] < num_fps[-2]:
                raise Exception("No valid cluster possible.")
        pbar.set_postfix({"Found clusters" : num_fps[-1] })
    num_fps = torch.as_tensor(num_fps)
    return x_fps, w_fps, num_fps
def disambiguate_fixed_points_adaptive(x_fp_list, weights, thresholds):
    pbar = tqdm(range(len(x_fp_list)))
    num_fps = [ ]
    x_fps = []
    w_fps = []
    for idx in pbar:
        found_threshold = False
        for threshold in thresholds:
            x_unique, x_unique_counts, labels = mhn_utils.group_by_distance(x_fp_list[idx], threshold)
            pbar.set_description(f"Threshold : {threshold}. Num. found {len(x_unique)}")
            if (len(num_fps) >= 1 and len(x_unique) >= num_fps[-1]) or len(num_fps) == 0:
                found_threshold = True
                w_unique = torch.zeros(x_unique.shape[0], weights.shape[-1])
                w_unique.index_add_(0, labels, weights[idx])
                w_unique /= x_unique_counts[:, None]
                num_fps.append(x_unique.shape[0])
                x_fps.append(torch.as_tensor(x_unique))
                w_fps.append(w_unique)
            if found_threshold:
                break
        if not found_threshold:
            raise Exception("No valid cluster possible.")
        pbar.set_postfix({"Found clusters" : num_fps[-1] })
    num_fps = torch.as_tensor(num_fps)
    return x_fps, w_fps, num_fps

### Select data

In [ ]:
DATASET = "mnist"
IS_IMAGE = DATASET in ["mnist", "stl10", "cifar10", "cifar100"]
data_path = Path("paper_results/experiments/")
quenched_files = list(data_path.glob(f"{DATASET}*weights_quenched*.pt"))
annealed_files = list(data_path.glob(f"{DATASET}*weights_annealed*.pt"))
file_df = {'is_quenched' : [], 'num_per_label' : [], 'noise' : [], 'is_euclidean' : [], 'is_centered' : [], 'file' : []}
for file in quenched_files + annealed_files:
    is_quenched = 'quenched' in file.stem
    num_per_label = re.search(r'per_label=(\d+)', file.stem)
    if num_per_label:
        num_per_label = int(num_per_label.group(1))
    else:
        num_per_label = np.nan
    is_euclidean = 'euclidean=True' in file.stem
    is_centered = 'centered=True' in file.stem
    logit_noise = re.search(r'noise=(\d+\.?\d*)', file.stem)
    if logit_noise:
        logit_noise = float(logit_noise.group(1))
    else:
        logit_noise = np.nan
    file_df['is_quenched'].append(is_quenched)
    file_df['num_per_label'].append(num_per_label)
    file_df['noise'].append(logit_noise)
    file_df['is_euclidean'].append(is_euclidean)
    file_df['is_centered'].append(is_centered)
    file_df['file'].append(str(file.absolute()))
file_df = pd.DataFrame.from_dict(file_df)

In [ ]:
is_centered = False
is_euclidean = True
sub_file_df = file_df[(file_df['is_euclidean'] == is_euclidean) & (file_df['is_centered'] == is_centered)]
logit_noise = 1.0
num_per_label = 1 if IS_IMAGE else None
if num_per_label is not None:
    quenched_file_mask = (file_df['is_euclidean'] == is_euclidean) & (file_df['is_centered'] == is_centered) & (file_df['is_quenched'] == True) & (file_df['num_per_label'] == num_per_label)
    annealed_file_mask = (file_df['is_euclidean'] == is_euclidean) & (file_df['is_centered'] == is_centered) & (file_df['is_quenched'] == False) & (file_df['num_per_label'] == num_per_label) & (file_df['noise'] == logit_noise)
else:
    quenched_file_mask = (file_df['is_euclidean'] == is_euclidean) & (file_df['is_centered'] == is_centered) & (file_df['is_quenched'] == True)
    annealed_file_mask = (file_df['is_euclidean'] == is_euclidean) & (file_df['is_centered'] == is_centered) & (file_df['is_quenched'] == False)  & (file_df['noise'] == logit_noise)

In [ ]:
quenched_file_name = file_df[quenched_file_mask]['file'].iloc[0]
quenched_data = torch.load(quenched_file_name)
quenched_weights = quenched_data['weights']
shift = quenched_data['shift']
rms = quenched_data['rms']
betas = quenched_data['betas']
biases = quenched_data['biases']
patterns = quenched_data['patterns']
gram = patterns @ patterns.T
quenched_x = quenched_weights @ patterns

In [ ]:
annealed_file_name = file_df[annealed_file_mask]['file'].iloc[0]
annealed_data = torch.load(annealed_file_name)
annealed_weights = annealed_data['weights']
annealed_x = annealed_weights @ patterns

### Extract unique fixed points (quenched)

In [ ]:
quenched_thresholds = [1e-2,1e-3,1e-4]
for quenched_threshold in quenched_thresholds:
    try:
        x_quenched_fps, w_quenched_fps, num_quenched_fps = disambiguate_fixed_points(quenched_x, quenched_weights, quenched_threshold, True)
        print(f"Proper clusters at {quenched_threshold}")
        break
    except:
        print(f"Threshold too small {quenched_threshold}")

In [ ]:
if IS_IMAGE:
    from diffusers import AutoencoderKL
    ae_model = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse")
    ae_model = ae_model.to(device).eval()
    ae_model.requires_grad_(False)
    ae_scaling = ae_model.config.scaling_factor
    
    source_data_file = Path(quenched_file_name).parent
    if num_per_label is not None or np.isfinite(num_per_label):
        source_data_file = source_data_file / f"{DATASET}_per_label={num_per_label}.pt"
    else:
        source_data_file = source_data_file / f"{DATASET}.pt"
    source_data = torch.load(source_data_file)
    source_latents = source_data['latents']
    C_latents, H_latents, W_latents = source_latents.shape[1:]

    image_quenched_fps = []
    for x in x_quenched_fps:
        torch.cuda.empty_cache()
        images = ae_model.decode((x*rms + shift).view(-1,C_latents, H_latents, W_latents).to(device)).sample.cpu()
        image_quenched_fps.append(images)

    #image_annealed_fps = []
    #for x in x_annealed_fps:
    #    torch.cuda.empty_cache()
    #    images = ae_model.decode(x.view(-1,C_latents, H_latents, W_latents).to(device)).sample.cpu()
    #    image_annealed_fps.append(images)
else:
    image_quenched_fps = None
    #image_annealed_fps = None

In [ ]:
#if image_annealed_fps is not None:
#    grayscale_conversion = torch.tensor([0.299, 0.587, 0.114])
#    jump_indices = torch.argwhere(torch.diff(num_annealed_fps) > 0).flatten()
#    for beta_idx in jump_indices:
#        images = image_annealed_fps[beta_idx]
#        fig, axs = plt.subplots(ncols=images.shape[0], figsize=(20,4))
#        if images.shape[0] == 1:
#            axs = [axs]
#        for idx in range(images.shape[0]):
#            axs[idx].imshow((1+(images[idx].permute(1,2,0) @ grayscale_conversion).clip(-1,1))/2)
#        plt.show()

In [ ]:
if image_quenched_fps is not None:
    grayscale_conversion = torch.tensor([0.299, 0.587, 0.114])
    jump_indices = torch.argwhere(torch.diff(num_quenched_fps) > 0).flatten()+1
    jump_indices = torch.cat([torch.zeros(1, dtype=jump_indices.dtype), jump_indices, (len(image_quenched_fps)-1)*torch.ones(1, dtype=jump_indices.dtype) ])
    for beta_idx in jump_indices:
        images = image_quenched_fps[beta_idx]
        fig, axs = plt.subplots(ncols=images.shape[0], figsize=(20,4))
        if images.shape[0] == 1:
            axs = [axs]
        for idx in range(images.shape[0]):
            axs[idx].imshow((1+(images[idx].permute(1,2,0) @ grayscale_conversion).clip(-1,1))/2)
        plt.show()

### PCA plots (quenched)

In [ ]:
fp_nodes = []
for beta_idx in range(len(x_quenched_fps)):
    for idx in range(len(x_quenched_fps[beta_idx])):
        fp_nodes.append((beta_idx, idx))
for cc_threshold in [1e-2]:
    
    edgelist_from, edgelist_to = distance_tree([x.numpy() for x in x_quenched_fps], cc_threshold)
    
    G = nx.Graph()
    G.add_nodes_from(fp_nodes)
    G.add_edges_from(
        zip(map(tuple, edgelist_from), map(tuple, edgelist_to))
    )
    
    fp_connected_components = list(nx.connected_components(G))
    
    num_fp_cc = len(fp_connected_components)
    fp_tags = {}
    for cc_index, cc_ in enumerate(fp_connected_components):
        cc = np.array(list(cc_))
        cc = cc[np.argsort(cc[:,0])]
        for el in cc:
            fp_tags[(el[0].item(),el[1].item())] = cc_index
    fp_tag_colors = np.array([
        mcolors.to_rgb(c)
        for c in glasbey.create_palette(palette_size=num_fp_cc)
    ])
    print(f"Number of connected components: {num_fp_cc}")

In [ ]:
pca_patterns = PCA(n_components=5).fit(patterns)
patterns_proj = pca_patterns.transform(patterns)
grayscale_conversion = torch.tensor([0.299, 0.587, 0.114])
jump_indices = torch.argwhere(torch.diff(num_quenched_fps) > 0).flatten()+1
jump_indices = torch.cat([torch.zeros(1, dtype=jump_indices.dtype), jump_indices, (len(num_quenched_fps)-1)*torch.ones(1, dtype=jump_indices.dtype) ])
for beta_idx in jump_indices:
    x_fp = x_quenched_fps[beta_idx]
    x_fp_proj = torch.as_tensor(pca_patterns.transform(x_fp))
    closest_fp_idx = ((patterns[:,None,:] - x_fp[None,:,:])**2).sum(dim=-1).argmin(dim=1)
    tags_fp = torch.as_tensor([ fp_tags[(beta_idx.item(),idx)] for idx in range(len(x_fp))]) 
    tags_patterns = torch.as_tensor([ fp_tags[(beta_idx.item(),idx.item())] for idx in closest_fp_idx])
    fig, axs = plt.subplots(ncols=2)
    axs[0].scatter(patterns_proj[:,0], patterns_proj[:,1],color=fp_tag_colors[tags_patterns], alpha=0.3)
    axs[0].scatter(x_fp_proj[:,0], x_fp_proj[:,1], color=fp_tag_colors[tags_fp], marker='s', s=50, edgecolors='black')
    for fp_idx in range(len(x_fp_proj)):
        axs[0].text(x_fp_proj[fp_idx,0], x_fp_proj[fp_idx,1], f"{tags_fp[fp_idx]}", fontsize=15 )
    axs[1].scatter(patterns_proj[:,2], patterns_proj[:,3],color=fp_tag_colors[tags_patterns], alpha=0.3)
    axs[1].scatter(x_fp_proj[:,2], x_fp_proj[:,3], color=fp_tag_colors[tags_fp], marker='s', s=50, edgecolors='black')
    axs[0].set_aspect('equal')
    axs[1].set_aspect('equal')
    plt.show()
    if IS_IMAGE:
        images = image_quenched_fps[beta_idx]
        fig, axs = plt.subplots(ncols=images.shape[0], figsize=(20,4))
        if images.shape[0] == 1:
            axs = [axs]
        for idx in range(images.shape[0]):
            axs[idx].imshow((1+(images[idx].permute(1,2,0) @ grayscale_conversion).clip(-1,1))/2)
            axs[idx].set_title(fp_tags[(beta_idx.item(),idx)])
        plt.show()